# 00 — Data Exploration
**Project:** Optimizer and Learning Rate Study of Ovarian Cancer Prediction  
**Course:** HI 192 — Knowledge Representation and Health Decision Support

This notebook explores the raw STRAMPN histopathological image dataset before any preprocessing is applied.

In [ ]:
import os
import pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns
from PIL import Image

RAW_DIR = pathlib.Path('../dataset/raw')
CLASSES = ['cancer', 'no_cancer']
RANDOM_SEED = 42

---
## 1. Dataset Overview

Count the total number of images per class and overall.

In [ ]:
# TODO: Count images per class and display a summary DataFrame
counts = {}
for cls in CLASSES:
    cls_dir = RAW_DIR / cls
    files = list(cls_dir.glob('*.*'))
    counts[cls] = len(files)

total = sum(counts.values())
summary = pd.DataFrame({
    'Class': list(counts.keys()),
    'Count': list(counts.values()),
    'Percentage': [f"{v/total*100:.1f}%" for v in counts.values()]
})
print(f"Total images: {total}")
display(summary)

---
## 2. Sample Image Visualization

Display a 2×5 grid of random sample images — 5 from each class — to inspect staining, quality, and visual variation.

In [ ]:
# TODO: Load and display 5 random samples per class in a 2×5 grid
import random
random.seed(RANDOM_SEED)

n_samples = 5
fig, axes = plt.subplots(len(CLASSES), n_samples, figsize=(15, 6))

for row_idx, cls in enumerate(CLASSES):
    cls_dir = RAW_DIR / cls
    files = sorted(cls_dir.glob('*.*'))
    samples = random.sample(files, min(n_samples, len(files)))
    for col_idx, fp in enumerate(samples):
        img = mpimg.imread(str(fp))
        axes[row_idx, col_idx].imshow(img)
        axes[row_idx, col_idx].axis('off')
        if col_idx == 0:
            axes[row_idx, col_idx].set_ylabel(cls, fontsize=12, fontweight='bold')

fig.suptitle('Sample Images per Class', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 3. Image Property Analysis

Inspect image dimensions, aspect ratios, and per-channel pixel statistics across the full dataset.

In [ ]:
# TODO: Gather width, height, and channel stats from every image
records = []
for cls in CLASSES:
    cls_dir = RAW_DIR / cls
    for fp in cls_dir.glob('*.*'):
        try:
            with Image.open(fp) as img:
                w, h = img.size
                mode = img.mode
                arr = np.array(img.convert('RGB'))
                records.append({
                    'class': cls,
                    'width': w,
                    'height': h,
                    'mode': mode,
                    'mean_R': arr[:,:,0].mean(),
                    'mean_G': arr[:,:,1].mean(),
                    'mean_B': arr[:,:,2].mean(),
                })
        except Exception as e:
            print(f'Could not read {fp}: {e}')

props = pd.DataFrame(records)
print("Dimension summary:")
display(props[['class','width','height']].groupby('class').describe())
print("\nMean pixel values per channel:")
display(props.groupby('class')[['mean_R','mean_G','mean_B']].mean().round(2))

In [ ]:
# TODO: Plot width vs height scatter coloured by class
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for cls in CLASSES:
    sub = props[props['class'] == cls]
    axes[0].scatter(sub['width'], sub['height'], alpha=0.4, label=cls, s=15)
axes[0].set_xlabel('Width (px)')
axes[0].set_ylabel('Height (px)')
axes[0].set_title('Image Dimensions')
axes[0].legend()

# Channel mean distributions
for ch, color in zip(['mean_R','mean_G','mean_B'], ['red','green','blue']):
    axes[1].hist(props[ch], bins=30, alpha=0.5, label=ch, color=color)
axes[1].set_xlabel('Mean Pixel Value')
axes[1].set_title('Per-Channel Pixel Distribution')
axes[1].legend()

plt.tight_layout()
plt.show()

---
## 4. Class Balance Check

Visualize the class distribution to assess whether the dataset is balanced or imbalanced, which informs augmentation strategy.

In [ ]:
# TODO: Bar chart of cancer vs no_cancer image counts
fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(
    counts.keys(),
    counts.values(),
    color=['#C0392B', '#2980B9'],
    edgecolor='black',
    width=0.5
)
for bar, val in zip(bars, counts.values()):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 3,
        str(val),
        ha='center', va='bottom', fontweight='bold'
    )
ax.set_title('Class Distribution — STRAMPN Dataset', fontweight='bold')
ax.set_xlabel('Class')
ax.set_ylabel('Image Count')
ax.set_ylim(0, max(counts.values()) * 1.15)
plt.tight_layout()
plt.show()

ratio = max(counts.values()) / min(counts.values())
print(f"Imbalance ratio (majority/minority): {ratio:.2f}")